# 14. Relevant Segment Extraction (RSE)

Return the best *contiguous run* of chunks instead of isolated top-k chunks.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Top-k returns *isolated* chunks, but a good answer often lives in a **run of consecutive chunks**. Relevant Segment Extraction (RSE) scores every chunk, subtracts a threshold so unrelated chunks become negative, then uses a maximum-sum subarray (Kadane) search to find the best contiguous segment(s). Segment length adapts to the question: short for a fact, long for an explanation.

## How it works

1. Score every chunk against the query
2. Subtract a threshold: irrelevant chunks become negative values
3. Find the contiguous run with the highest total value
4. Mask it out and repeat for the next best segment
5. Send each segment as one context

```text
 chunk value (score - threshold):   -.1  -.2  +.3  +.4  +.2  -.1  -.3
                                               [  best segment  ]
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

Score each chunk, subtract a threshold so irrelevant chunks are negative, then find the maximum-sum contiguous segment (Kadane's algorithm).

In [ ]:
import numpy as np

def best_segments(query, threshold=0.25, max_len=4, n_segments=2):
    scores = store.vecs @ embed(query)[0] - threshold
    segments, used = [], np.zeros(len(scores), dtype=bool)
    for _ in range(n_segments):
        best = (0.0, None)
        for i in range(len(scores)):
            total = 0.0
            for j in range(i, min(i + max_len, len(scores))):
                if used[j]:
                    break
                total += scores[j]
                if total > best[0]:
                    best = (total, (i, j))
        if best[1] is None:
            break
        i, j = best[1]
        used[i:j + 1] = True
        segments.append((best[0], i, j))
    return segments

query = "How do reranking and hybrid search improve retrieval?"
segs = best_segments(query)
for score, i, j in segs:
    print(f"chunks {i}-{j}  score={score:.2f}")

In [ ]:
context = [" ".join(chunks[i:j + 1]) for _, i, j in segs]
print(answer(query, context))

## Inspect the scores
Plot the per-chunk values so you can see why a segment was chosen.

In [ ]:
import matplotlib.pyplot as plt
q = "How do reranking and hybrid search improve retrieval?"
vals = store.vecs @ embed(q)[0] - 0.25
plt.figure(figsize=(11, 3)); plt.bar(range(len(vals)), vals, color=["#4c78a8" if v > 0 else "#bbb" for v in vals])
plt.axhline(0, color="k", lw=.8); plt.xlabel("chunk index"); plt.ylabel("score - threshold"); plt.tight_layout(); plt.show()

## Evaluate
A segment counts as a hit if its combined text contains the evidence.

In [ ]:
def rse_search(q, k=3):
    segs = best_segments(q, n_segments=k)
    return [{"idx": i, "text": " ".join(chunks[i:j + 1]), "score": s, "meta": {}} for s, i, j in segs]

compare_retrievers({"top-k chunks": store.search, "relevant segments": rse_search}, k=2)

## Threshold sensitivity
The threshold sets how selective you are: low = long, inclusive segments; high = short, precise ones.

In [ ]:
import pandas as pd
rows = []
for thr in [0.1, 0.2, 0.3, 0.4]:
    segs = best_segments(q, threshold=thr)
    rows.append({"threshold": thr, "segments": [(i, j) for _, i, j in segs], "total_chunks": sum(j - i + 1 for _, i, j in segs)})
pd.DataFrame(rows).set_index("threshold")

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Returns coherent passages, not scattered snippets | Needs ordered chunks from a single document |
| Segment length adapts to the question | Threshold requires tuning |
| Cheap: only arithmetic on scores | Long segments can inflate the prompt |

## When to use it

Long single documents (reports, books, transcripts) where answers span several adjacent chunks.

## Experiments to try

- Cap `max_len` at 2 and 8 and compare answers.
- Add a small bonus for chunks adjacent to high scorers.
- Apply per document across a multi-document corpus.

## Key takeaways

- Turn scores into signed values, then solve max-sum subarray.
- The threshold is the main knob.
- Contiguity gives the LLM natural context.

---
**Next:** [15. Contextual Compression](../15_contextual_compression/contextual_compression.ipynb)